# Phase 3 · pandas and Spark ETL

Compare repeated CSV → Parquet runs from the aggregate benchmark report.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from fraud_detection.paths import find_project_root

artifact = find_project_root() / "artifacts/phase_03/benchmark.json"
with artifact.open(encoding="utf-8") as stream:
    report = json.load(stream)
if not report["completed"]:
    raise ValueError("Benchmark is incomplete")

display(pd.Series(report["environment"], name="Environment").to_frame())
print("Source SHA-256:", report["source_csv_sha256"])
print("Code revision:", report["revision"])

In [ ]:
rows = []
for result in report["results"]:
    for engine, summary in result["summary"].items():
        rows.append({
            "scale": result["scale"],
            "engine": engine,
            "rows": result["parity"]["rows"],
            "input_mib": result["input_csv_bytes"] / 1024**2,
            "median_seconds": summary["median_elapsed_seconds"],
            "stdev_seconds": summary["sample_stdev_seconds"],
            "peak_rss_mib": max(
                item["peak_rss_bytes_sampled"] for item in result["measurements"][engine]
            ) / 1024**2,
        })
summary = pd.DataFrame(rows)
display(summary)
axes = summary.pivot(index="scale", columns="engine", values="median_seconds").plot.bar(
    ylabel="Median ETL time (seconds)", xlabel="Input scale"
)
axes.set_title(f"CSV → Parquet · {report['environment']['spark_master']}")
plt.show()

The timer includes CSV reading, deterministic transformation, and Snappy Parquet writing. The results reflect this workload and host.